# Notebook 09 — Adapter Inspection, Merging, and Export

Training an adapter is only half the workflow. You also need to know what changed, how many parameters were trainable, how to save and reload the adapter, when to merge it, and what a deployment handoff should include.

## What you will build

This notebook walks through the full adapter lifecycle:

1. run a tiny local adapter finetuning job
2. inspect trainable parameters and LoRA tensors
3. save the adapter separately
4. reload it onto a clean base model
5. test a merge path conservatively
6. build an export manifest and handoff checklist

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## Step 1: Define notebook paths and helper functions

The helper functions focus on deterministic inspection, parameter accounting, and artifact validation.

In [ ]:
import gc
from peft import PeftModel

random.seed(9)

ARTIFACT_DIR = Path(OUTPUT_DIR) / "notebook_09_adapter_inspection"
ADAPTER_DIR = ARTIFACT_DIR / "adapter_separate"
MERGED_DIR = ARTIFACT_DIR / "merged_export"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def render_chat(record, active_tokenizer):
    messages = [
        {"role": "system", "content": "You are Castalia Mentor, a concise tutor focused on practical finetuning workflows."},
        {"role": "user", "content": record["user"]},
        {"role": "assistant", "content": record["assistant"]},
    ]
    return active_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def count_parameters(active_model):
    total_params = sum(param.numel() for param in active_model.parameters())
    trainable_params = sum(param.numel() for param in active_model.parameters() if param.requires_grad)
    return total_params, trainable_params

def generate_completion(active_model, active_tokenizer, prompt, max_new_tokens=90):
    messages = [
        {"role": "system", "content": "You are Castalia Mentor, a concise tutor focused on practical finetuning workflows."},
        {"role": "user", "content": prompt},
    ]
    try:
        FastLanguageModel.for_inference(active_model)
    except Exception:
        pass
    input_ids = active_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(active_model.device)
    outputs = active_model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )
    completion_ids = outputs[0][input_ids.shape[-1]:]
    return active_tokenizer.decode(completion_ids, skip_special_tokens=True).strip()

def collect_lora_rows(active_model):
    rows = []
    for name, param in active_model.named_parameters():
        if "lora_A" in name or "lora_B" in name:
            rows.append(
                {
                    "parameter_name": name,
                    "shape": tuple(param.shape),
                    "numel": int(param.numel()),
                    "requires_grad": bool(param.requires_grad),
                    "abs_mean": float(param.detach().abs().mean().cpu()),
                }
            )
    return pd.DataFrame(rows)

def collect_trainable_rows(active_model):
    rows = []
    for name, param in active_model.named_parameters():
        if param.requires_grad:
            rows.append(
                {
                    "parameter_name": name,
                    "numel": int(param.numel()),
                    "module_family": next((module for module in ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] if module in name), "other"),
                }
            )
    return pd.DataFrame(rows)

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Artifacts:", ARTIFACT_DIR.resolve())

## Step 2: Create a tiny local dataset

The dataset is deliberately small because the notebook is about adapter lifecycle and inspection, not quality maximization.

In [ ]:
examples = [
    {"id": "01", "user": "Why do teams save adapters separately?", "assistant": "Separate adapters are smaller, easier to version, and easier to roll back without touching the base model."},
    {"id": "02", "user": "When is merging useful?", "assistant": "Merging is useful when deployment wants one standalone checkpoint instead of a base-plus-adapter pair."},
    {"id": "03", "user": "Why verify trainable parameter counts?", "assistant": "Trainable parameter counts confirm that the PEFT setup matched your intent and did not accidentally unfreeze the wrong weights."},
    {"id": "04", "user": "What does inspecting LoRA weights tell you?", "assistant": "Inspecting LoRA weights tells you where updates live, how large they are, and whether the saved adapter matches the configured target modules."},
    {"id": "05", "user": "Why reload an adapter onto a clean base model?", "assistant": "Reloading verifies that the saved artifact is portable beyond the original training session."},
    {"id": "06", "user": "Why keep an export checklist?", "assistant": "An export checklist reduces handoff mistakes by making sure tokenizer files, base-model references, and validation notes travel with the artifact."},
    {"id": "07", "user": "How should merged and separate artifacts be compared?", "assistant": "Compare them with the same prompts and decoding settings so any difference is easy to spot."},
    {"id": "08", "user": "Why is adapter-first still the practical default?", "assistant": "Adapter-first training keeps finetuning cheap, reversible, and easy to iterate on with open-source tooling."},
]

dataset = Dataset.from_list(examples)
formatted_dataset = Dataset.from_list([
    {**row, "text": render_chat(row, tokenizer)}
    for row in dataset.to_list()
])

audit_prompts = [
    "Why do teams often keep adapters separate?",
    "Why reload an adapter onto a clean base model?",
    "When is merging useful?",
]

display(pd.DataFrame(examples)[["id", "user"]])

## Step 3: Record clean-base outputs first

The baseline must come from a fresh base model, not the adapterized training object already created by the shared setup cell.

In [ ]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)
base_tokenizer.padding_side = "right"

base_rows = []
for prompt in audit_prompts:
    base_rows.append({"prompt": prompt, "base_completion": generate_completion(base_model, base_tokenizer, prompt)})

base_outputs_df = pd.DataFrame(base_rows)
display(base_outputs_df)

## Step 4: Inspect the adapter configuration before training

Parameter accounting is easier when you know the intended target modules and nominal rank up front.

In [ ]:
peft_config = next(iter(model.peft_config.values()))
adapter_config_df = pd.DataFrame(
    [
        {"field": "r", "value": peft_config.r},
        {"field": "lora_alpha", "value": peft_config.lora_alpha},
        {"field": "lora_dropout", "value": peft_config.lora_dropout},
        {"field": "bias", "value": peft_config.bias},
        {"field": "target_modules", "value": ", ".join(peft_config.target_modules)},
    ]
)
display(adapter_config_df)

## Step 5: Define the tiny training run

We keep the training budget short because the purpose is to create a portable artifact we can inspect end to end.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(ARTIFACT_DIR / "trainer_run"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=2,
    max_steps=12,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    save_steps=12,
    save_total_limit=1,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    report_to="none",
    seed=3407,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=training_args,
)

training_args

## Step 6: Train the adapter

This produces the weights we will inspect, save, reload, and possibly merge.

In [ ]:
train_result = trainer.train()
train_log_df = pd.DataFrame(trainer.state.log_history)
display(train_log_df.tail())

## Step 7: Count total versus trainable parameters

This is the fastest sanity check that the run really stayed parameter efficient.

In [ ]:
total_params, trainable_params = count_parameters(model)
param_summary_df = pd.DataFrame(
    [
        {"metric": "total_parameters", "value": int(total_params)},
        {"metric": "trainable_parameters", "value": int(trainable_params)},
        {"metric": "trainable_percent", "value": round(100 * trainable_params / total_params, 4)},
    ]
)
display(param_summary_df)

## Step 8: Break trainable parameters down by module family

This turns one percentage into a clearer explanation of where the adapter actually lives.

In [ ]:
trainable_rows_df = collect_trainable_rows(model)
trainable_by_family_df = trainable_rows_df.groupby("module_family", as_index=False)["numel"].sum().sort_values("numel", ascending=False)
display(trainable_by_family_df)

## Step 9: Inspect the LoRA tensors directly

You should see A and B matrices corresponding to the targeted projection layers.

In [ ]:
lora_df = collect_lora_rows(model).sort_values("parameter_name").reset_index(drop=True)
display(lora_df.head(20))
print("LoRA tensor rows:", len(lora_df))

## Step 10: Summarize LoRA tensor statistics

This is a compact check that the saved adapter is non-empty and shaped the way you expect.

In [ ]:
lora_stats_df = pd.DataFrame(
    [
        {"metric": "tensor_rows", "value": int(len(lora_df))},
        {"metric": "mean_abs_value", "value": round(float(lora_df["abs_mean"].mean()), 6)},
        {"metric": "max_abs_value", "value": round(float(lora_df["abs_mean"].max()), 6)},
        {"metric": "total_lora_numel", "value": int(lora_df["numel"].sum())},
    ]
)
display(lora_stats_df)

## Step 11: Compare base and tuned outputs before saving

This gives us a direct behavior check before any save or reload step enters the picture.

In [ ]:
tuned_rows = []
for prompt in audit_prompts:
    tuned_rows.append({"prompt": prompt, "tuned_completion": generate_completion(model, tokenizer, prompt)})

tuned_outputs_df = pd.DataFrame(tuned_rows)
base_vs_tuned_df = base_outputs_df.merge(tuned_outputs_df, on="prompt", how="left")
display(base_vs_tuned_df)

## Step 12: Save the adapter as a separate artifact

Keeping the adapter separate is still the operational default because it preserves rollback and variant flexibility.

In [ ]:
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

saved_adapter_files = sorted(path.name for path in ADAPTER_DIR.iterdir())
saved_adapter_files_df = pd.DataFrame({"saved_file": saved_adapter_files})
display(saved_adapter_files_df)

## Step 13: Verify that the saved files include the essentials

An adapter export is incomplete if the tokenizer or adapter config is missing.

In [ ]:
artifact_check_df = pd.DataFrame(
    [
        {"item": "adapter_config.json", "present": (ADAPTER_DIR / "adapter_config.json").exists()},
        {"item": "adapter weights", "present": any(path.name.startswith("adapter_model") for path in ADAPTER_DIR.iterdir())},
        {"item": "tokenizer_config.json", "present": (ADAPTER_DIR / "tokenizer_config.json").exists()},
        {"item": "special_tokens_map.json", "present": (ADAPTER_DIR / "special_tokens_map.json").exists()},
    ]
)
display(artifact_check_df)

## Step 14: Reload the adapter onto a clean base model

This is the core portability test.

In [ ]:
reload_base_model, reload_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)
reload_tokenizer.padding_side = "right"

reloaded_model = PeftModel.from_pretrained(reload_base_model, str(ADAPTER_DIR))

reloaded_rows = []
for prompt in audit_prompts:
    reloaded_rows.append({"prompt": prompt, "reloaded_completion": generate_completion(reloaded_model, reload_tokenizer, prompt)})

reloaded_outputs_df = pd.DataFrame(reloaded_rows)
reload_compare_df = base_vs_tuned_df.merge(reloaded_outputs_df, on="prompt", how="left")
reload_compare_df["tuned_matches_reloaded"] = reload_compare_df["tuned_completion"] == reload_compare_df["reloaded_completion"]
display(reload_compare_df)

## Step 15: Measure reload fidelity

If the tuned and reloaded outputs align closely, the saved artifact is behaving the way a handoff expects.

In [ ]:
reload_fidelity_df = pd.DataFrame(
    [
        {"metric": "reload_match_rate", "value": float(reload_compare_df["tuned_matches_reloaded"].mean())},
        {"metric": "prompt_count", "value": int(len(reload_compare_df))},
    ]
)
display(reload_fidelity_df)

## Step 16: Attempt a merge path conservatively

On a quantized Colab workflow, merge can fail or be inconvenient. That is fine. The point is to test the path and document the result.

In [ ]:
merge_status = {"status": "not_attempted", "message": ""}
merged_outputs_df = pd.DataFrame(columns=["prompt", "merged_completion"])

try:
    merged_model = reloaded_model.merge_and_unload()
    merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
    reload_tokenizer.save_pretrained(str(MERGED_DIR))

    merged_rows = []
    for prompt in audit_prompts:
        merged_rows.append({"prompt": prompt, "merged_completion": generate_completion(merged_model, reload_tokenizer, prompt)})

    merged_outputs_df = pd.DataFrame(merged_rows)
    merge_status = {"status": "merged", "message": "Merged model saved successfully."}
except Exception as exc:
    merge_status = {"status": "kept_separate", "message": str(exc)}

merge_status

## Step 17: Compare base, tuned, reloaded, and merged outputs

This is the most useful table for workflow validation because it shows which artifact paths preserve behavior.

In [ ]:
final_compare_df = reload_compare_df.merge(merged_outputs_df, on="prompt", how="left")
display(final_compare_df)

## Step 18: Summarize merge availability

A failed merge is not a notebook failure. It often just means the deployment handoff should ship the separate adapter instead.

In [ ]:
merge_summary_df = pd.DataFrame(
    [
        {"metric": "merge_status", "value": merge_status["status"]},
        {"metric": "merged_dir_exists", "value": MERGED_DIR.exists()},
        {"metric": "message", "value": merge_status["message"][:160]},
    ]
)
display(merge_summary_df)

## Step 19: Build an export manifest

The manifest is a compact machine-readable handoff artifact.

In [ ]:
export_manifest = {
    "base_model_name": MODEL_NAME,
    "adapter_dir": str(ADAPTER_DIR),
    "merged_dir": str(MERGED_DIR) if MERGED_DIR.exists() else None,
    "trainable_parameters": int(trainable_params),
    "trainable_percent": float(round(100 * trainable_params / total_params, 4)),
    "merge_status": merge_status,
    "reload_match_rate": float(reload_compare_df["tuned_matches_reloaded"].mean()),
}

manifest_path = ARTIFACT_DIR / "export_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as handle:
    json.dump(export_manifest, handle, indent=2)

export_manifest

## Step 20: Build a deployment handoff checklist

This is the human-facing version of the manifest.

In [ ]:
deployment_checklist_df = pd.DataFrame(
    [
        {"item": "Base model reference recorded", "status": True, "notes": MODEL_NAME},
        {"item": "Adapter directory saved", "status": ADAPTER_DIR.exists(), "notes": str(ADAPTER_DIR)},
        {"item": "Tokenizer files saved", "status": (ADAPTER_DIR / "tokenizer_config.json").exists(), "notes": "required for prompt formatting compatibility"},
        {"item": "Parameter summary captured", "status": True, "notes": "see param_summary_df"},
        {"item": "Reload test completed", "status": True, "notes": f"match rate = {float(reload_compare_df['tuned_matches_reloaded'].mean()):.2f}"},
        {"item": "Merge path tested", "status": True, "notes": merge_status["status"]},
        {"item": "Merged export available", "status": MERGED_DIR.exists(), "notes": str(MERGED_DIR) if MERGED_DIR.exists() else merge_status["message"][:120]},
    ]
)
display(deployment_checklist_df)

## Step 21: Add a deployment decision table

This table answers the practical question: when do you ship the adapter separately and when do you hand off a merged checkpoint?

In [ ]:
deployment_decision_df = pd.DataFrame(
    [
        {"situation": "Need easy rollback or multiple task variants", "ship_format": "separate adapter", "reason": "smaller artifact and easier swap-in"},
        {"situation": "Serving stack expects one checkpoint", "ship_format": "merged model if merge succeeds", "reason": "simpler runtime loading path"},
        {"situation": "Quantized Colab merge is awkward", "ship_format": "separate adapter", "reason": "do not force a brittle merge workflow"},
        {"situation": "Final offline export on larger hardware", "ship_format": "merged model can be reasonable", "reason": "merge cost moves out of the notebook loop"},
    ]
)
display(deployment_decision_df)

## Step 22: Save a notebook summary

This captures the key outcome in one place for later notebooks or deployment notes.

In [ ]:
notebook_summary = {
    "separate_adapter_is_default": True,
    "reload_fidelity": float(reload_compare_df["tuned_matches_reloaded"].mean()),
    "merge_status": merge_status["status"],
    "manifest_path": str(manifest_path),
}

with open(ARTIFACT_DIR / "notebook_summary.json", "w", encoding="utf-8") as handle:
    json.dump(notebook_summary, handle, indent=2)

notebook_summary

## Step 23: Release extra models

The notebook has loaded several model objects. Cleaning up is good Colab hygiene before moving on.

In [ ]:
del base_model
del base_tokenizer
del reload_base_model
del reload_tokenizer
if "merged_model" in globals():
    del merged_model
clear_gpu()
print("Released auxiliary models from memory.")

## Key takeaways

- Inspecting adapters confirms what was actually trained.
- Trainable-parameter accounting is the fastest PEFT sanity check.
- Separate adapters remain the practical default for reversible open-source workflows.
- Reload testing is mandatory before handoff.
- Merge should be treated as a tested option, not an assumed requirement.